<a href="https://colab.research.google.com/github/aishanee-sinha/Multi-Agent-Autonomous-Workforce-Assistant/blob/Leela8256-codes/pipeline_email.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers peft bitsandbytes icalendar python-dateutil accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import icalendar
from datetime import datetime
import dateutil.parser
import sys
import os

# --- 1. Helper Function: Load Model ---
def load_trained_model(base_model_id, adapter_path):
    """Loads the base model and applies the LoRA adapters."""

    if not os.path.exists(adapter_path):
        print(f"Error: Adapter path not found: {adapter_path}", file=sys.stderr)
        print("Please ensure the model was trained and saved to this location.", file=sys.stderr)
        return None, None

    print(f"Loading base model: {base_model_id}...")
    try:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

        tokenizer = AutoTokenizer.from_pretrained(base_model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb,
            device_map="auto",
        )

        print(f"Applying LoRA adapters from: {adapter_path}...")
        model = PeftModel.from_pretrained(base_model, adapter_path)
        model = model.eval()
        print("✅ Model loaded and ready.")
        return model, tokenizer
    except Exception as e:
        print(f"Error loading model: {e}", file=sys.stderr)
        import traceback
        traceback.print_exc()
        return None, None

# --- 2. Helper Function: Prompt & Inference ---
def format_inference_prompt(email_text):
    """Formats the raw email text into the Mistral prompt template."""
    instruction = (
        "You are a precise calendar event extraction assistant. "
        "Extract the calendar event. Return ONLY a JSON object with keys: "
        "title, start_utc, end_utc, location, attendees, notes.\n"
        "- All timestamps must be ISO8601 with timezone or Z.\n"
        "- If time is ambiguous, set start_utc and end_utc to null and explain in notes.\n"
        "- Attendees should include emails from From, To, and Cc."
    )
    prompt = f"[INST] {instruction.strip()}\n\nHere is the email:\n\n{email_text.strip()} [/INST]"
    return prompt

def run_inference(model, tokenizer, email_text, max_new_tokens=1024):
    """Runs inference and attempts to parse the JSON output."""
    prompt = format_inference_prompt(email_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    try:
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start != -1 and json_end != -1:
            json_str = response_text[json_start:json_end]
            return json.loads(json_str), response_text
        else:
            return None, response_text
    except json.JSONDecodeError:
        print(f"\nWarning: Failed to parse JSON from output:\n{response_text}\n", file=sys.stderr)
        return None, response_text

# --- 3. Helper Function: ICS Conversion ---
def convert_json_to_ics(event_json):
    """Converts the model's JSON output to an .ics file string."""
    cal = icalendar.Calendar()
    cal.add('prodid', '-//Calendar Extraction Model//')
    cal.add('version', '2.0')
    event = icalendar.Event()

    event.add('summary', event_json.get('title', 'No Title'))
    event.add('description', event_json.get('notes', ''))

    loc = event_json.get('location')
    if loc:
        event.add('location', loc)

    try:
        start_utc = event_json.get('start_utc')
        end_utc = event_json.get('end_utc')

        if start_utc and end_utc:
            dt_start = dateutil.parser.isoparse(start_utc)
            dt_end = dateutil.parser.isoparse(end_utc)
            event.add('dtstart', dt_start)
            event.add('dtend', dt_end)
        elif start_utc: # Handle if only start_utc is provided
            dt_start = dateutil.parser.isoparse(start_utc)
            event.add('dtstart', dt_start)
            event.add('dtend', dt_start + datetime.timedelta(hours=1)) # Default 1 hour
        else:
            event.add('dtstart', datetime.now().date())

    except Exception as e:
        print(f"Error parsing datetime: {e}. Skipping times.", file=sys.stderr)
        event.add('dtstart', datetime.now().date()) # Fallback

    attendees = event_json.get('attendees', [])
    for email in attendees:
        attendee = icalendar.vCalAddress(f'MAILTO:{email}')
        event.add('attendee', attendee, encode=0)

    cal.add_component(event)
    return cal.to_ical().decode('utf-8')

# --- 4. Main Execution ---
def main_test():
    BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
    ADAPTER_PATH = "./calendar_extraction_model_final"

    # Load the model
    model, tokenizer = load_trained_model(BASE_MODEL, ADAPTER_PATH)

    if model is None or tokenizer is None:
        print("Exiting due to model loading failure.")
        return

    # ** New dummy email with start and end time **
    # Note: "tomorrow" will be interpreted relative to your system's current date.
    dummy_email_text = (
        "Subject: Q4 Planning Session\n"
        "From: leelaprasad.dammalapati@sjsu.edu\n"
        "To: aishanee.sinha@sjsu.edu,debika.choudhury@sjsu.edu\n"
        "Cc: \n"
        "Bcc: \n\n"
        "Body:\n"
        "Hi all,\n\n"
        "Following up on our last discussion, let's discuss the Q4 planning session. lets meet on 7th novemner 2025\n\n"
        "Lets discuss Q4.\n\n"
        "Please confirm.\n\n"
        "Best,\n"
        "Leela"
    ).strip()

    print("="*80)
    print(f"--- INPUT EMAIL ---\n{dummy_email_text}\n{'-'*20}")

    # Run inference
    predicted_json, raw_output = run_inference(model, tokenizer, dummy_email_text)

    if predicted_json:
        print(f"--- PARSED JSON ---\n{json.dumps(predicted_json, indent=2)}\n{'-'*20}")

        # Convert to ICS
        ics_string = convert_json_to_ics(predicted_json)

        print(f"--- GENERATED .ICS FILE ---\n{ics_string}\n{'-'*20}")

        # Save to file
        output_filename = "dummy_meeting_with_end_time.ics"
        with open(output_filename, "w") as f:
            f.write(ics_string)
        print(f"✅ Successfully saved to {output_filename}")
        print("You can now import this file into your calendar.")

    else:
        print("❌ Model did not produce valid JSON.")
        print(f"--- RAW MODEL OUTPUT ---\n{raw_output}")

# --- This makes the script runnable ---
if __name__ == "__main__":
    main_test()

Loading base model: mistralai/Mistral-7B-Instruct-v0.2...


Loading checkpoint shards: 100%|██████████| 3/3 [00:11<00:00,  3.99s/it]


Applying LoRA adapters from: ./calendar_extraction_model_final...
✅ Model loaded and ready.
--- INPUT EMAIL ---
Subject: Q4 Planning Session
From: leelaprasad.dammalapati@sjsu.edu
To: aishanee.sinha@sjsu.edu,debika.choudhury@sjsu.edu
Cc: 
Bcc: 

Body:
Hi all,

Following up on our last discussion, let's discuss the Q4 planning session. lets meet on 7th novemner 2025

Lets discuss Q4.

Please confirm.

Best,
Leela
--------------------
--- PARSED JSON ---
{
  "title": "Q4 Planning Session",
  "start_utc": null,
  "end_utc": null,
  "location": null,
  "attendees": [
    "aishanee.sinha@sjsu.edu",
    "debika.choudhury@sjsu.edu",
    "leelaprasad.dammalapati@sjsu.edu"
  ],
  "notes": "Ambiguous or missing date/time. Hi all,\n\nFollowing up on our last discussion, let's discuss the Q4 planning session. lets meet on 7th novemner 2025\n\nLets discuss Q4.\n\nPlease confirm.\n\nBest,\nLeela"
}
--------------------
--- GENERATED .ICS FILE ---
BEGIN:VCALENDAR
VERSION:2.0
PRODID:-//Calendar Extrac